# FIPS Risk Lookup (Notebook Dev Tool)

Notebook version of the standalone lookup tool. It reads FEMA NRI state CSV files from `plugins/emfn-risk-assessment-plugin/assets/data/` and renders county hazard results directly in notebook output.

## Usage
1. Run all cells in order.
2. Enter a 5-digit county FIPS code.
3. Click **Lookup Risk Data** (or use one of the sample buttons).

In [ ]:
from pathlib import Path
import csv
from html import escape
from IPython.display import HTML, display

RISK_THRESHOLD = 50.0
DATA_DIR = Path.cwd().parent / "plugins" / "emfn-risk-assessment-plugin" / "assets" / "data"

NRI_HAZARD_LABELS = {
    "avln": "Avalanche",
    "cfld": "Coastal Flooding",
    "cwav": "Cold Wave",
    "drgt": "Drought",
    "erqk": "Earthquake",
    "hail": "Hail",
    "hwav": "Heat Wave",
    "hrcn": "Hurricane",
    "istm": "Ice Storm",
    "lnds": "Landslide",
    "ltng": "Lightning",
    "ifld": "Inland Flooding",
    "swnd": "Strong Wind",
    "trnd": "Tornado",
    "tsun": "Tsunami",
    "vlcn": "Volcanic Activity",
    "wfir": "Wildfire",
    "wntw": "Winter Weather",
}

STATE_CODES = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA", "08": "CO", "09": "CT",
    "10": "DE", "11": "DC", "12": "FL", "13": "GA", "15": "HI", "16": "ID", "17": "IL",
    "18": "IN", "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME", "24": "MD",
    "25": "MA", "26": "MI", "27": "MN", "28": "MS", "29": "MO", "30": "MT", "31": "NE",
    "32": "NV", "33": "NH", "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
    "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI", "45": "SC", "46": "SD",
    "47": "TN", "48": "TX", "49": "UT", "50": "VT", "51": "VA", "53": "WA", "54": "WV",
    "55": "WI", "56": "WY",
}

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")

print(f"Using data directory: {DATA_DIR}")

In [ ]:
def parse_county_row_for_fips(csv_path: Path, target_fips: str):
    with csv_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        if "county_fips" not in (reader.fieldnames or []):
            raise ValueError(f"county_fips column not found in {csv_path.name}")

        for row in reader:
            if row.get("county_fips", "").strip() == target_fips:
                return row
    return None


def extract_hazards(row: dict, threshold: float = RISK_THRESHOLD):
    hazards = []
    for key, value in row.items():
        if not key.endswith("_risk_score"):
            continue

        try:
            score = float(value)
        except (TypeError, ValueError):
            continue

        if score < threshold:
            continue

        hazard_code = key.replace("_risk_score", "").lower()
        label = NRI_HAZARD_LABELS.get(hazard_code)
        if label:
            hazards.append({"label": label, "score": score})

    return sorted(hazards, key=lambda h: h["score"], reverse=True)


def render_result(row: dict, fips: str, threshold: float = RISK_THRESHOLD):
    county_name = escape((row.get("county") or "Unknown County").strip())
    state_name = escape((row.get("state") or "Unknown State").strip())
    hazards = extract_hazards(row, threshold=threshold)

    chips = "".join(
        f"<span style='display:inline-block;background:#fff;border:1px solid #ddd;border-radius:6px;padding:6px 12px;margin:4px 6px 4px 0;font-size:14px;'>"
        f"{escape(h['label'])} <strong style='color:#d9534f;'>({round(h['score'])}%)</strong></span>"
        for h in hazards
    )

    body = chips if chips else (
        f"<div style='background:#e3f2fd;border:1px solid #90caf9;color:#1565c0;padding:12px;border-radius:8px;'>"
        f"No high-risk hazards found (all scores below {threshold:g}%).</div>"
    )

    return HTML(
        f"<div style='background:#f9f9f9;border:1px solid #e2e2e2;border-radius:10px;padding:18px;margin-top:8px;'>"
        f"<div style='border-bottom:2px solid #e2e2e2;padding-bottom:12px;margin-bottom:12px;'>"
        f"<div style='font-size:20px;font-weight:600;color:#111;'>{county_name}</div>"
        f"<div style='color:#666;font-size:16px;'>{state_name}</div>"
        f"<div style='color:#888;font-size:13px;margin-top:6px;'>FIPS: <code>{escape(fips)}</code></div>"
        f"</div>"
        f"<div><strong>FEMA's >= {threshold:g}% Risk Hazards:</strong></div>"
        f"<div style='margin-top:8px;'>{body}</div>"
        f"</div>"
    )


def lookup_fips(fips: str):
    fips = (fips or "").strip()
    if len(fips) != 5 or not fips.isdigit():
        return HTML("<div style='background:#fff3f3;border:1px solid #ffcdd2;color:#c62828;padding:12px;border-radius:8px;'>Please enter a valid 5-digit FIPS code.</div>")

    state_code = STATE_CODES.get(fips[:2])
    if not state_code:
        return HTML("<div style='background:#fff3f3;border:1px solid #ffcdd2;color:#c62828;padding:12px;border-radius:8px;'>Invalid state code in FIPS. First 2 digits must be a valid state FIPS code.</div>")

    csv_path = DATA_DIR / f"{state_code}.csv"
    if not csv_path.exists():
        return HTML(f"<div style='background:#fff3f3;border:1px solid #ffcdd2;color:#c62828;padding:12px;border-radius:8px;'>CSV file not found: {escape(str(csv_path))}</div>")

    try:
        row = parse_county_row_for_fips(csv_path, fips)
    except Exception as exc:
        return HTML(f"<div style='background:#fff3f3;border:1px solid #ffcdd2;color:#c62828;padding:12px;border-radius:8px;'>Error reading CSV: {escape(str(exc))}</div>")

    if not row:
        return HTML(f"<div style='background:#fff3f3;border:1px solid #ffcdd2;color:#c62828;padding:12px;border-radius:8px;'>County FIPS {escape(fips)} not found in {state_code}.csv</div>")

    return render_result(row, fips=fips)

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if widgets is None:
    print("ipywidgets is not installed. Run lookup manually, for example: display(lookup_fips('13121'))")
else:
    fips_input = widgets.Text(
        value="",
        placeholder="e.g., 13121",
        description="FIPS:",
        layout=widgets.Layout(width="360px"),
    )

    lookup_btn = widgets.Button(description="Lookup Risk Data", button_style="primary")
    sample_btns = [
        widgets.Button(description="13121 Fulton, GA"),
        widgets.Button(description="06037 Los Angeles, CA"),
        widgets.Button(description="48201 Harris, TX"),
        widgets.Button(description="12086 Miami-Dade, FL"),
    ]
    sample_values = ["13121", "06037", "48201", "12086"]

    out = widgets.Output()

    def run_lookup(*_):
        with out:
            out.clear_output()
            display(lookup_fips(fips_input.value))

    lookup_btn.on_click(run_lookup)

    for btn, value in zip(sample_btns, sample_values):
        def _handler(_, v=value):
            fips_input.value = v
            run_lookup()
        btn.on_click(_handler)

    display(widgets.HTML("<h3 style='margin:0 0 8px 0;'>FIPS Risk Lookup</h3>"))
    display(widgets.HBox([fips_input, lookup_btn]))
    display(widgets.HBox(sample_btns))
    display(out)

In [ ]:
# Non-widget fallback example:
display(lookup_fips("13121"))